<a href="https://colab.research.google.com/github/xyaoaf/UrbanTreeStudywithLidar/blob/main/230403_spatialCorrelation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rasterio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from rasterio.plot import show
import seaborn as sns
# set the maximum number of columns to display to 30
pd.set_option('display.max_columns', 80)
from sklearn.linear_model import LinearRegression

In [ ]:
e_folder_30 = "/content/drive/MyDrive/XihanYAO_GISDataShare/230402_East30StatisticsLayers"
e_folder_2 = "/content/drive/MyDrive/XihanYAO_GISDataShare/230402_East2StatisticsLayers"
w_folder_30 = "/content/drive/MyDrive/XihanYAO_GISDataShare/230402_West30StatisticsLayers"
w_folder_2 = "/content/drive/MyDrive/XihanYAO_GISDataShare/230402_West2StatisticsLayers"
lstpath = "/content/drive/MyDrive/XihanYAO_GISDataShare/230102_CityScaleAnalysis(LST,NDVI,GNDVI,EVI,SAVI,VegUmixing,NDBI,NDWI)/landsatImageStack2019-08-05_230109.tif"

# lst

In [ ]:
lst = rasterio.open(lstpath)
crs = lst.crs
print (crs)
# Get the names of all bands
band_names = lst.descriptions
# Print the names of all bands
print(band_names)

In [ ]:
# Display the raster
fig, ax = plt.subplots()
lst_band = lst.read(1)
im = ax.imshow(lst.read(1), cmap='gray', interpolation='none')
ax.set_title("LST")
fig.colorbar(im, ax=ax)
plt.show()

# West Area

In [ ]:
file_path = w_folder_30 + "/area.tif"
src = rasterio.open(file_path)

In [ ]:
# Read the first band (layer) of the raster
band = src.read(1)

# Get the 5th and 95th percentiles of the pixel values
p5, p95 = np.percentile(band, (5, 95))

# Get the file name without the extension
file_name = os.path.splitext(os.path.basename(file_path))[0]

# Get some information about the raster
print(f"Raster shape: {src.shape}")
print(f"Raster CRS: {src.crs}")
print(f"Raster extent: {src.bounds}")

# Display the raster
fig, ax = plt.subplots()
im = ax.imshow(band, cmap='gray', vmin=p5, vmax=p95, interpolation='none')
ax.set_title(file_name)
fig.colorbar(im, ax=ax)
plt.show()

In [ ]:
# Get the geospatial coordinates of the center of each pixel
indices = np.where(band > 600)
x, y = rasterio.transform.xy(src.transform, indices[0], indices[1], offset='center')

# Create a DataFrame with the pixel coordinates and geospatial coordinates
df = pd.DataFrame({'x': indices[1], 'y': indices[0], 'lon': x, 'lat': y})

# Get the file name without the extension
file_name = os.path.splitext(os.path.basename(file_path))[0]

# Get some information about the raster
print(f"Raster shape: {src.shape}")
print(f"Raster CRS: {src.crs}")
print(f"Raster extent: {src.bounds}")

# Display the raster
fig, ax = plt.subplots()
im = ax.imshow(band, cmap='gray', vmin=0, vmax=1000, interpolation='none')
ax.set_title(file_name)
fig.colorbar(im, ax=ax)

# Plot the coordinates of the selected pixels as red dots
ax.plot(df['x'], df['y'], 'ro', markersize=1)

plt.show()

# Print the DataFrame with the pixel and geospatial coordinates
print(df)

In [ ]:
# # Define a function to extract the raster value at a given point
# def extract_raster_value(number, point, src):
#     x, y = point['lon'], point['lat']
#     row, col = src.read(number).index(x, y)
#     return src.read(number, window=((row, row+1), (col, col+1)))[0][0]

In [ ]:
# Extract raster values at coordinates
values0 = []
values2 = []
values3 = []
values4 = []
values5 = []
values6 = []
values7 = []
values8 = []
values9 = []
for index, row in df.iterrows():
    x, y = row['lon'], row['lat']
    value0 = next(lst.sample([(x, y)]))[0]
    values0.append(value0)
    value2 = next(lst.sample([(x, y)]))[2]
    values2.append(value2)
    value3 = next(lst.sample([(x, y)]))[3]
    values3.append(value3)
    value4 = next(lst.sample([(x, y)]))[4]
    values4.append(value4)
    value5 = next(lst.sample([(x, y)]))[5]
    values5.append(value5)
    value6 = next(lst.sample([(x, y)]))[6]
    values6.append(value6)
    value7 = next(lst.sample([(x, y)]))[7]
    values7.append(value7)
    value8 = next(lst.sample([(x, y)]))[8]
    values8.append(value8)
    value9 = next(lst.sample([(x, y)]))[9]
    values9.append(value9)

# Add raster values as a new column in the dataframe
df['LST'] = values0
df['ndviLandsat'] = values2
df['gndviLandsat'] = values3
df['eviLandsat'] = values4
df['saviLandsat'] = values5
df['unmixingVegLandsat'] = values6
df['ndmiLandsat'] = values7
df['mndwiLandsat'] = values8
df['tasseledGreenness'] = values9
df

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
# df.plot(ax=ax, color='red',markersize = 10, marker='o',
#         legend = True, legend_kwds={'label':'Sampled Points','shrink': 0.3})
extent=[lst.bounds[0], lst.bounds[2], lst.bounds[1], lst.bounds[3]]
ax = rasterio.plot.show(lst.read(1), extent=extent, ax=ax, cmap='Spectral_r')
im = ax.get_images()[0]
fig.colorbar(im, ax=ax,orientation = 'horizontal')

# Add the points to the plot
ax.scatter(df['lon'], df['lat'], color='red', marker='o', s=50)

In [ ]:
# Loop through each raster layer in the folder
for raster_file in sorted(os.listdir(w_folder_30)):
    # Check if file is a raster layer
    if raster_file.endswith(".tif"):
        # Open raster layer
        raster = rasterio.open(os.path.join(w_folder_30, raster_file))

        # Extract raster values at coordinates
        values = []
        for index, row in df.iterrows():
            x, y = row['lon'], row['lat']
            value = next(raster.sample([(x, y)]))[0]
            values.append(value)

        # Add raster values as a new column in the dataframe
        column_name = os.path.splitext(raster_file)[0] # Use filename as column name
        df[column_name] = values

In [ ]:
df

In [ ]:
# get the correlation between column 5 and all the other columns since column 6
corr = df.iloc[:, 5:].corrwith(df['LST'])

# loop through each column and create a scatter plot with correlation line and R-squared value
for i in range(len(corr)):
    if not pd.isna(corr[i]):
        fig, ax = plt.subplots()
        sns.regplot(x=df.iloc[:, i+5], y=df['LST'], ax=ax)
        ax.set_xlabel(df.columns[i+5])
        ax.set_ylabel('LST')
        corr_label = 'Correlation: {:.2f}'.format(corr[i])
        r2_label = 'R-squared: {:.2f}'.format(corr[i]**2)
        ax.annotate(corr_label + '\n' + r2_label, xy=(0.05, 0.85), xycoords='axes fraction')
        plt.show()

In [ ]:
# Print out the column names
print(df.columns)

In [ ]:
# Define the dependent variable
y = df['LST']

# Define the independent variables
x_cols = ['m0_6', 'm1_2', 'm1_8', 'm2_4', 'm3', 'm6','m9','m13','m18','m24','m32','m40']  # Replace with your list of column names
X = df[x_cols]

# Fit the linear regression model
model = LinearRegression().fit(X, y)
# Print the coefficients
print('Coefficients:', model.coef_)
# Print the intercept
print('Intercept:', model.intercept_)
# Print the R-squared value
print('R-squared:', model.score(X, y))

In [ ]:
# Define the dependent variable
y = df['LST']

# Define the independent variables
x_cols = ['zq5','zq10', 'zq15', 'zq20', 'zq25', 'zq30', 'zq35','zq40','zq45','zq50','zq55','zq60','zq65','zq70','zq75','zq80','zq85','zq90']  # Replace with your list of column names
X = df[x_cols]

# Fit the linear regression model
model = LinearRegression().fit(X, y)
# Print the coefficients
print('Coefficients:', model.coef_)
# Print the intercept
print('Intercept:', model.intercept_)
# Print the R-squared value
print('R-squared:', model.score(X, y))

In [ ]:
# export the dataframe to a csv file
df.to_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/230102_CityScaleAnalysis(LST,NDVI,GNDVI,EVI,SAVI,VegUmixing,NDBI,NDWI)/230403LSTRegressionW.csv', index=False)

# East Area

In [ ]:
file_path = e_folder_30 + "/area.tif"
srcE = rasterio.open(file_path)

In [ ]:
# Read the first band (layer) of the raster
band = srcE.read(1)

# Get the 5th and 95th percentiles of the pixel values
p5, p95 = np.percentile(band, (5, 95))

# Get the file name without the extension
file_name = os.path.splitext(os.path.basename(file_path))[0]

# Get some information about the raster
print(f"Raster shape: {srcE.shape}")
print(f"Raster CRS: {srcE.crs}")
print(f"Raster extent: {srcE.bounds}")

# Display the raster
fig, ax = plt.subplots()
im = ax.imshow(band, cmap='gray', vmin=p5, vmax=p95, interpolation='none')
ax.set_title(file_name)
fig.colorbar(im, ax=ax)
plt.show()

In [ ]:
# Get the geospatial coordinates of the center of each pixel
indices = np.where(band > 600)
x, y = rasterio.transform.xy(srcE.transform, indices[0], indices[1], offset='center')

# Create a DataFrame with the pixel coordinates and geospatial coordinates
dfE = pd.DataFrame({'x': indices[1], 'y': indices[0], 'lon': x, 'lat': y})

# Get the file name without the extension
file_name = os.path.splitext(os.path.basename(file_path))[0]

# Get some information about the raster
print(f"Raster shape: {srcE.shape}")
print(f"Raster CRS: {srcE.crs}")
print(f"Raster extent: {srcE.bounds}")

# Display the raster
fig, ax = plt.subplots()
im = ax.imshow(band, cmap='gray', vmin=0, vmax=1000, interpolation='none')
ax.set_title(file_name)
fig.colorbar(im, ax=ax)

# Plot the coordinates of the selected pixels as red dots
ax.plot(dfE['x'], dfE['y'], 'ro', markersize=1)

plt.show()

# Print the DataFrame with the pixel and geospatial coordinates
print(dfE)

In [ ]:
# # Define a function to extract the raster value at a given point
# def extract_raster_value(point, srcE):
#     x, y = point['lon'], point['lat']
#     row, col = srcE.read(1).index(x, y)
#     return srcE.read(1, window=((row, row+1), (col, col+1)))[0][0]

In [ ]:
# Extract raster values at coordinates
values0 = []
values2 = []
values3 = []
values4 = []
values5 = []
values6 = []
values7 = []
values8 = []
values9 = []
for index, row in dfE.iterrows():
    x, y = row['lon'], row['lat']
    value0 = next(lst.sample([(x, y)]))[0]
    values0.append(value0)
    value2 = next(lst.sample([(x, y)]))[2]
    values2.append(value2)
    value3 = next(lst.sample([(x, y)]))[3]
    values3.append(value3)
    value4 = next(lst.sample([(x, y)]))[4]
    values4.append(value4)
    value5 = next(lst.sample([(x, y)]))[5]
    values5.append(value5)
    value6 = next(lst.sample([(x, y)]))[6]
    values6.append(value6)
    value7 = next(lst.sample([(x, y)]))[7]
    values7.append(value7)
    value8 = next(lst.sample([(x, y)]))[8]
    values8.append(value8)
    value9 = next(lst.sample([(x, y)]))[9]
    values9.append(value9)

# Add raster values as a new column in the dataframe
dfE['LST'] = values0
dfE['ndviLandsat'] = values2
dfE['gndviLandsat'] = values3
dfE['eviLandsat'] = values4
dfE['saviLandsat'] = values5
dfE['unmixingVegLandsat'] = values6
dfE['ndmiLandsat'] = values7
dfE['mndwiLandsat'] = values8
dfE['tasseledGreenness'] = values9
dfE

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
# dfE.plot(ax=ax, color='red',markersize = 10, marker='o',
#         legend = True, legend_kwds={'label':'Sampled Points','shrink': 0.3})
extent=[lst.bounds[0], lst.bounds[2], lst.bounds[1], lst.bounds[3]]
ax = rasterio.plot.show(lst.read(1), extent=extent, ax=ax, cmap='Spectral_r')
im = ax.get_images()[0]
fig.colorbar(im, ax=ax,orientation = 'horizontal')

# Add the points to the plot
ax.scatter(dfE['lon'], dfE['lat'], color='red', marker='o', s=50)

In [ ]:
# Loop through each raster layer in the folder
for raster_file in sorted(os.listdir(e_folder_30)):
    # Check if file is a raster layer
    if raster_file.endswith(".tif"):
        # Open raster layer
        raster = rasterio.open(os.path.join(e_folder_30, raster_file))

        # Extract raster values at coordinates
        values = []
        for index, row in dfE.iterrows():
            x, y = row['lon'], row['lat']
            value = next(raster.sample([(x, y)]))[0]
            values.append(value)

        # Add raster values as a new column in the dataframe
        column_name = os.path.splitext(raster_file)[0] # Use filename as column name
        dfE[column_name] = values

In [ ]:
dfE

In [ ]:
# get the correlation between column 5 and all the other columns since column 6
corr = dfE.iloc[:, 5:].corrwith(dfE['LST'])

# loop through each column and create a scatter plot with correlation line and R-squared value
for i in range(len(corr)):
    if not pd.isna(corr[i]):
        fig, ax = plt.subplots()
        sns.regplot(x=dfE.iloc[:, i+5], y=dfE['LST'], ax=ax)
        ax.set_xlabel(dfE.columns[i+5])
        ax.set_ylabel('LST')
        corr_label = 'Correlation: {:.2f}'.format(corr[i])
        r2_label = 'R-squared: {:.2f}'.format(corr[i]**2)
        ax.annotate(corr_label + '\n' + r2_label, xy=(0.05, 0.85), xycoords='axes fraction')
        plt.show()

In [ ]:
# Print out the column names
print(dfE.columns)

In [ ]:
# Define the dependent variable
y = dfE['LST']

# Define the independent variables
x_cols = ['m0_6', 'm1_2', 'm1_8', 'm2_4', 'm3', 'm6','m9','m13','m18','m24','m32','m40']  # Replace with your list of column names
X = dfE[x_cols]

# Fit the linear regression model
model = LinearRegression().fit(X, y)
# Print the coefficients
print('Coefficients:', model.coef_)
# Print the intercept
print('Intercept:', model.intercept_)
# Print the R-squared value
print('R-squared:', model.score(X, y))

In [ ]:
# Define the dependent variable
y = dfE['LST']

# Define the independent variables
x_cols = ['zq5','zq10', 'zq15', 'zq20', 'zq25', 'zq30', 'zq35','zq40','zq45','zq50','zq55','zq60','zq65','zq70','zq75','zq80','zq85','zq90']  # Replace with your list of column names
X = dfE[x_cols]

# Fit the linear regression model
model = LinearRegression().fit(X, y)
# Print the coefficients
print('Coefficients:', model.coef_)
# Print the intercept
print('Intercept:', model.intercept_)
# Print the R-squared value
print('R-squared:', model.score(X, y))

In [ ]:
# export the dataframe to a csv file
dfE.to_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/230102_CityScaleAnalysis(LST,NDVI,GNDVI,EVI,SAVI,VegUmixing,NDBI,NDWI)/230403LSTRegressionE.csv', index=False)

# Combined

In [ ]:
# Concatenate the dataframes and add a new column
dfAll = pd.concat([df, dfE])
dfAll['origin'] = ['W'] * len(df) + ['E'] * len(dfE)
# Reorder the columns
dfAll = dfAll.reindex(columns=['x', 'y', 'origin'] + list(dfAll.columns[2:-1]))

print(dfAll)

In [ ]:
# Define color map for categories
category_colors = {'W': 'orange', 'E': 'blue'}

# get the correlation between column 5 and all the other columns since column 6
corr = dfAll.iloc[:, 6:].corrwith(dfAll['LST'])

# loop through each column and create a scatter plot with correlation line and R-squared value
for i in range(len(corr)):
    if not pd.isna(corr[i]):
        fig, ax = plt.subplots()
        
        # Add color to scatter plot based on category column
        for category, color in category_colors.items():
            ax.scatter(x=dfAll.loc[dfAll['origin']==category].iloc[:, i+6], 
                       y=dfAll.loc[dfAll['origin']==category]['LST'], 
                       color=color, 
                       alpha=0.5, 
                       label=category)
        
        ax.set_xlabel(dfAll.columns[i+6])
        ax.set_ylabel('LST')
        corr_label = 'Correlation: {:.2f}'.format(corr[i])
        r2_label = 'R-squared: {:.2f}'.format(corr[i]**2)
        ax.annotate(corr_label + '\n' + r2_label, xy=(0.05, 0.85), xycoords='axes fraction')
        ax.legend()
        plt.show()


In [ ]:
# Initialize an empty dictionary to store R-squared values
r2_values = {}

# Get the correlation between column 5 and all the other columns since column 6
corr = dfAll.iloc[:, 6:].corrwith(dfAll['LST'])

# Loop through each column and create a scatter plot with correlation line and R-squared value
for i in range(len(corr)):
    if not pd.isna(corr[i]):
        fig, ax = plt.subplots()
        category_colors = {'W': 'orange', 'E': 'blue'}
        colors = [category_colors[c] for c in dfAll['origin']]
        sns.regplot(x=dfAll.iloc[:, i+6], y=dfAll['LST'], ax=ax, scatter_kws={'color': colors})
        ax.set_xlabel(dfAll.columns[i+6])
        ax.set_ylabel('LST')
        corr_label = 'Correlation: {:.2f}'.format(corr[i])
        r2_label = 'R-squared: {:.2f}'.format(corr[i]**2)
        ax.annotate(corr_label + '\n' + r2_label, xy=(0.05, 0.85), xycoords='axes fraction')
        
        # Store the R-squared value in the dictionary with the column name as the key
        r2_values[dfAll.columns[i+6]] = corr[i]**2
        
        plt.show()
# create a bar chart from the r2_values dictionary
plt.bar(r2_values.keys(), r2_values.values())
# set the x-axis label
plt.xlabel('Column')
# set the y-axis label
plt.ylabel('R-squared value')
# set the chart title
plt.title('R-squared values for LST correlation')
# display the chart
plt.show()

In [ ]:
columns_to_select = ['LST','origin','m0_6', 'm1_2', 'm1_8', 'm2_4', 'm3', 'm6','m9','m13','m18','m24','m32','m40']

# Create a sub dataframe with the selected columns
df_profile = dfAll.loc[:, columns_to_select]
df_profile['m_sum'] = df_profile.loc[:, 'm0_6':'m40'].sum(axis=1)
df_profile

In [ ]:
# Initialize an empty dictionary to store R-squared values
r2_values = {}

# Get the correlation between column 5 and all the other columns since column 6
corr = df_profile.iloc[:, 2:].corrwith(df_profile['LST'])

# Loop through each column and create a scatter plot with correlation line and R-squared value
for i in range(len(corr)):
    if not pd.isna(corr[i]):
        fig, ax = plt.subplots()
        category_colors = {'W': 'orange', 'E': 'blue'}
        colors = [category_colors[c] for c in df_profile['origin']]
        sns.regplot(x=df_profile.iloc[:, i+2], y=df_profile['LST'], ax=ax, scatter_kws={'color': colors})
        ax.set_xlabel(df_profile.columns[i+2])
        ax.set_ylabel('LST')
        corr_label = 'Correlation: {:.2f}'.format(corr[i])
        r2_label = 'R-squared: {:.2f}'.format(corr[i]**2)
        ax.annotate(corr_label + '\n' + r2_label, xy=(0.05, 0.85), xycoords='axes fraction')
        
        # Store the R-squared value in the dictionary with the column name as the key
        r2_values[df_profile.columns[i+2]] = corr[i]**2
        
        plt.show()
# create a bar chart from the r2_values dictionary
plt.bar(r2_values.keys(), r2_values.values())
# set the x-axis label
plt.xlabel('Column')
# set the y-axis label
plt.ylabel('R-squared value')
# set the chart title
plt.title('R-squared values for LST correlation')
# display the chart
plt.show()

In [ ]:
columns_to_select = ['LST','origin','zq5','zq10', 'zq15', 'zq20', 'zq25', 'zq30', 'zq35','zq40','zq45','zq50','zq55','zq60','zq65','zq70','zq75','zq80','zq85','zq90','zq95']

# Create a sub dataframe with the selected columns
df_zq = dfAll.loc[:, columns_to_select]
df_zq

In [ ]:
# Initialize an empty dictionary to store R-squared values
r2_values = {}

# Get the correlation between column 5 and all the other columns since column 6
corr = df_zq.iloc[:, 2:].corrwith(df_zq['LST'])

# Loop through each column and create a scatter plot with correlation line and R-squared value
for i in range(len(corr)):
    if not pd.isna(corr[i]):
        fig, ax = plt.subplots()
        category_colors = {'W': 'orange', 'E': 'blue'}
        colors = [category_colors[c] for c in df_zq['origin']]
        sns.regplot(x=df_zq.iloc[:, i+2], y=df_zq['LST'], ax=ax, scatter_kws={'color': colors})
        ax.set_xlabel(df_zq.columns[i+2])
        ax.set_ylabel('LST')
        corr_label = 'Correlation: {:.2f}'.format(corr[i])
        r2_label = 'R-squared: {:.2f}'.format(corr[i]**2)
        ax.annotate(corr_label + '\n' + r2_label, xy=(0.05, 0.85), xycoords='axes fraction')
        
        # Store the R-squared value in the dictionary with the column name as the key
        r2_values[df_zq.columns[i+2]] = corr[i]**2
        
        plt.show()
# create a bar chart from the r2_values dictionary
plt.bar(r2_values.keys(), r2_values.values())
# set the x-axis label
plt.xlabel('Column')
# set the y-axis label
plt.ylabel('R-squared value')
# set the chart title
plt.title('R-squared values for LST correlation')
# display the chart
plt.show()

In [ ]:
columns_to_select = ['LST','origin','zpcum1','zpcum2', 'zpcum3', 'zpcum4', 'zpcum5', 'zpcum6', 'zpcum7','zpcum8','zpcum9']

# Create a sub dataframe with the selected columns
df_zpcum = dfAll.loc[:, columns_to_select]
df_zpcum

In [ ]:
# Initialize an empty dictionary to store R-squared values
r2_values = {}

# Get the correlation between column 5 and all the other columns since column 6
corr = df_zpcum.iloc[:, 2:].corrwith(df_zpcum['LST'])

# Loop through each column and create a scatter plot with correlation line and R-squared value
for i in range(len(corr)):
    if not pd.isna(corr[i]):
        fig, ax = plt.subplots()
        category_colors = {'W': 'orange', 'E': 'blue'}
        colors = [category_colors[c] for c in df_zpcum['origin']]
        sns.regplot(x=df_zpcum.iloc[:, i+2], y=df_zpcum['LST'], ax=ax, scatter_kws={'color': colors})
        ax.set_xlabel(df_zpcum.columns[i+2])
        ax.set_ylabel('LST')
        corr_label = 'Correlation: {:.2f}'.format(corr[i])
        r2_label = 'R-squared: {:.2f}'.format(corr[i]**2)
        ax.annotate(corr_label + '\n' + r2_label, xy=(0.05, 0.85), xycoords='axes fraction')
        
        # Store the R-squared value in the dictionary with the column name as the key
        r2_values[df_zpcum.columns[i+2]] = corr[i]**2
        
        plt.show()
# create a bar chart from the r2_values dictionary
plt.bar(r2_values.keys(), r2_values.values())
# set the x-axis label
plt.xlabel('Column')
# set the y-axis label
plt.ylabel('R-squared value')
# set the chart title
plt.title('R-squared values for LST correlation')
# display the chart
plt.show()

In [ ]:
columns_to_select = ['LST','origin','p1th', 'p2th', 'p3th', 'p4th','p5th']

# Create a sub dataframe with the selected columns
df_pth = dfAll.loc[:, columns_to_select]
df_pth

In [ ]:
# Initialize an empty dictionary to store R-squared values
r2_values = {}

# Get the correlation between column 5 and all the other columns since column 6
corr = df_pth.iloc[:, 2:].corrwith(df_pth['LST'])

# Loop through each column and create a scatter plot with correlation line and R-squared value
for i in range(len(corr)):
    if not pd.isna(corr[i]):
        fig, ax = plt.subplots()
        category_colors = {'W': 'orange', 'E': 'blue'}
        colors = [category_colors[c] for c in df_pth['origin']]
        sns.regplot(x=df_pth.iloc[:, i+2], y=df_pth['LST'], ax=ax, scatter_kws={'color': colors})
        ax.set_xlabel(df_pth.columns[i+2])
        ax.set_ylabel('LST')
        corr_label = 'Correlation: {:.2f}'.format(corr[i])
        r2_label = 'R-squared: {:.2f}'.format(corr[i]**2)
        ax.annotate(corr_label + '\n' + r2_label, xy=(0.05, 0.85), xycoords='axes fraction')
        
        # Store the R-squared value in the dictionary with the column name as the key
        r2_values[df_pth.columns[i+2]] = corr[i]**2
        
        plt.show()
# create a bar chart from the r2_values dictionary
plt.bar(r2_values.keys(), r2_values.values())
# set the x-axis label
plt.xlabel('Column')
# set the y-axis label
plt.ylabel('R-squared value')
# set the chart title
plt.title('R-squared values for LST correlation')
# display the chart
plt.show()

In [ ]:
# Define the dependent variable
y = dfAll['LST']

# Define the independent variables
x_cols = ['m0_6', 'm1_2', 'm1_8', 'm2_4', 'm3', 'm6','m9','m13','m18','m24','m32','m40']  # Replace with your list of column names
X = dfAll[x_cols]

# Fit the linear regression model
model = LinearRegression().fit(X, y)
# Print the coefficients
print('Coefficients:', model.coef_)
# Print the intercept
print('Intercept:', model.intercept_)
# Print the R-squared value
print('R-squared:', model.score(X, y))

In [ ]:
# Define the dependent variable
y = dfAll['LST']

# Define the independent variables
x_cols = ['zq5','zq10', 'zq15', 'zq20', 'zq25', 'zq30', 'zq35','zq40','zq45','zq50','zq55','zq60','zq65','zq70','zq75','zq80','zq85','zq90','zq95']  # Replace with your list of column names
X = dfAll[x_cols]

# Fit the linear regression model
model = LinearRegression().fit(X, y)
# Print the coefficients
print('Coefficients:', model.coef_)
# Print the intercept
print('Intercept:', model.intercept_)
# Print the R-squared value
print('R-squared:', model.score(X, y))

In [ ]:
# Define the dependent variable
y = dfAll['LST']

# Define the independent variables
x_cols = ['zpcum1','zpcum2','zpcum3','zpcum4','zpcum5','zpcum6','zpcum7','zpcum8','zpcum9']  # Replace with your list of column names
X = dfAll[x_cols]

# Fit the linear regression model
model = LinearRegression().fit(X, y)
# Print the coefficients
print('Coefficients:', model.coef_)
# Print the intercept
print('Intercept:', model.intercept_)
# Print the R-squared value
print('R-squared:', model.score(X, y))

In [ ]:
# Define the dependent variable
y = dfAll['LST']

# Define the independent variables
x_cols = ['p1th','p2th','p3th','p4th','p5th']  # Replace with your list of column names
X = dfAll[x_cols]

# Fit the linear regression model
model = LinearRegression().fit(X, y)
# Print the coefficients
print('Coefficients:', model.coef_)
# Print the intercept
print('Intercept:', model.intercept_)
# Print the R-squared value
print('R-squared:', model.score(X, y))